# Aufgabe 6d: CellCNN mit quadratischer Filterantwort

Der Hinweis aus Aufgabe 6 wird durch eine gezielte Erweiterung der linearen Filterantwort aufgegriffen:

`r_k(x) = ReLU(b_k + sum_j(w_kj * x_j) + sum_j(q_kj * x_j²))`.

Die Eingaben sind wie bisher ausschließlich anhand innerer Trainingsspender standardisierte `arcsinh(x/5)`-Markerwerte. Pro Filter kommen 37 Gewichte hinzu; keine Marker-Kreuzprodukte, Hidden Layers oder neue Hyperparametersuche. Die zusätzlichen Gewichte starten exakt bei null und erhalten dieselbe L2-Strafe wie die linearen Filtergewichte. Bei null gesetzten Zusatzgewichten entspricht das Modell der Baseline einschließlich Initialisierung und Top-1%-Mean-Pooling.

Ein negativer quadratischer Beitrag kann zusammen mit einem linearen Beitrag einen bevorzugten Markerbereich ausdrücken. Das ist eine Hypothese für eine bessere Populationserkennung, keine Garantie für bessere Klassifikation. Quadratische Terme können auch Ausreißer und Überanpassung verstärken.

Vor diesem Lauf werden die zehn vorhandenen Splits **0–9** festgelegt. Alle zehn gehen unabhängig vom Ergebnis in den Vergleich ein. Das Notebook lädt die dazugehörigen Task-4-Baseline-Modelle und Predictions; es trainiert die Baseline nicht erneut. Bestehende Notebooks und Artefakte bleiben unverändert. Neue Ausgaben tragen ausschließlich den Präfix `task6_quadratic_`.

In [1]:
from pathlib import Path
import copy
import csv
import hashlib
import json
import os
import sys
import time
import warnings
from importlib.metadata import version

import flowkit as fk
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

BONUS_SPLIT_IDS = list(range(10))
SPLIT_IDS = BONUS_SPLIT_IDS
GATE, RUN_MODE, GATE_SUFFIX = "gated_alive", "full", "_alive"
COFACTOR, TOP_FRACTION = 5.0, 0.01
LEARNING_RATE, L2_COEFFICIENT = 0.01, 1e-4
TRAINING_CELLS_PER_INPUT, TRAINING_INPUTS_PER_DONOR = 3000, 200
PREDICTION_CELLS_PER_INPUT, PREDICTION_INPUTS_PER_DONOR = 20000, 5
SCALER_CELLS_PER_DONOR = 20000
FILTER_COUNTS, BATCH_SIZE = [3, 4, 5], 128
MAX_EPOCHS, EARLY_STOPPING_PATIENCE = 100, 5
EVALUATION_CELLS_PER_DONOR, EVALUATION_SEED = 20000, 63000
EXPECTED_EVENT_COUNTS = {"gated_alive": 3438750}

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "NK_cell_dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "NK_cell_dataset/NK_cell_dataset"
FCS_DIR = DATA_ROOT / "NK_cell_dataset" / GATE
LABELS_PATH, MARKERS_PATH = DATA_ROOT / "NK_fcs_samples_with_labels.csv", DATA_ROOT / "NK_markers.csv"
TABLES = PROJECT_ROOT / "results/tables"
SPLITS_PATH = TABLES / "task4_donor_splits.csv"
sys.path.insert(0, str(PROJECT_ROOT))
from src.task4_artifacts import make_run_config, file_digest, validate_prediction_splits, validate_parameter_table
from src.task5_interpretation import SavedCellCNN, restore_scaler

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Gerät: {DEVICE}; PyTorch: {torch.__version__}; Bonus-Splits: {BONUS_SPLIT_IDS}")


Gerät: cuda; PyTorch: 2.14.0+cu130; Bonus-Splits: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [2]:
# Zweck: Alle Spenderdaten transformiert, aber weiterhin spenderweise getrennt laden.
with MARKERS_PATH.open(newline="", encoding="utf-8-sig") as stream:
    markers = next(csv.reader(stream))

label_table = pd.read_csv(LABELS_PATH)
label_table["donor_id"] = label_table["fcs_filename"].str.replace(
    r"_NK\.fcs$", "", regex=True
)
label_table["label"] = label_table["label"].astype(int)
fcs_paths = sorted(FCS_DIR.glob("*.fcs"))
fcs_table = pd.DataFrame(
    {
        "donor_id": [path.stem.removesuffix(GATE_SUFFIX) for path in fcs_paths],
        "fcs_path": fcs_paths,
    }
)
sample_table = (
    label_table[["donor_id", "label"]]
    .merge(fcs_table, on="donor_id", validate="one_to_one")
    .sort_values("donor_id")
    .reset_index(drop=True)
)
donor_splits = pd.read_csv(SPLITS_PATH)

assert len(markers) == 37
assert len(fcs_paths) == 20
assert len(sample_table) == 20
assert set(SPLIT_IDS).issubset(set(donor_splits["split_id"]))
split_labels = donor_splits[["donor_id", "label"]].drop_duplicates()
assert split_labels.merge(
    sample_table[["donor_id", "label"]],
    on=["donor_id", "label"],
    validate="one_to_one",
).shape[0] == len(sample_table)

def load_transformed_fcs(path: Path) -> np.ndarray:
    """Eine FCS-Datei read-only als ArcSinh-transformierte Markermatrix laden."""
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=r"FCS file .* reported incorrect data offset.*"
        )
        sample = fk.Sample(str(path), ignore_offset_error=True)
    frame = sample.as_dataframe(source="raw")
    short_names = pd.Index(sample.pns_labels, name="marker")
    if not short_names.is_unique:
        raise ValueError(f"Doppelte FCS-Kurznamen in {path.name}.")
    frame.columns = short_names
    missing = [marker for marker in markers if marker not in frame.columns]
    if missing:
        raise ValueError(f"Fehlende Marker in {path.name}: {missing}")
    values = frame.loc[:, markers].to_numpy(dtype=np.float32, copy=True)
    values = np.arcsinh(values / COFACTOR).astype(np.float32, copy=False)
    if not np.isfinite(values).all():
        raise ValueError(f"Nicht-endliche Werte in {path.name}.")
    return values

# Die getrennte Ablage verhindert ein versehentliches Aufteilen eines Spenders.
data_by_donor = {
    row.donor_id: load_transformed_fcs(row.fcs_path)
    for row in sample_table.itertuples(index=False)
}
label_by_donor = sample_table.set_index("donor_id")["label"].to_dict()
assert sum(map(len, data_by_donor.values())) == EXPECTED_EVENT_COUNTS[GATE]

display(
    pd.DataFrame(
        {
            "donor_id": list(data_by_donor),
            "label": [label_by_donor[x] for x in data_by_donor],
            "cells": [len(data_by_donor[x]) for x in data_by_donor],
        }
    )
)


,donor_id,label,cells
0,a_001,1,82324
1,a_002,1,108267
2,a_003,0,97529
3,a_004,0,140687
4,a_005,1,155335
5,a_006,0,90075
6,a_007,1,104805
7,a_009,0,216632
8,a_010,0,122209
9,a_011,0,258461


In [3]:
# Referenzcode und Eingabedaten prüfen, ohne 04c auszuführen oder Artefakte zu schreiben.
baseline_path = TABLES / "task4_cellcnn_predictions_gated_alive_full.csv"
baseline_config_path = baseline_path.with_suffix(".config.json")
baseline_config = json.loads(baseline_config_path.read_text())
expected_parameters = {
    "method": "cellcnn", "gate": GATE, "run_mode": RUN_MODE,
    "cofactor": COFACTOR, "learning_rate": LEARNING_RATE,
    "l2_coefficient": L2_COEFFICIENT, "top_fraction": TOP_FRACTION,
    "training_cells_per_input": TRAINING_CELLS_PER_INPUT,
    "training_inputs_per_donor": TRAINING_INPUTS_PER_DONOR,
    "prediction_cells_per_input": PREDICTION_CELLS_PER_INPUT,
    "prediction_inputs_per_donor": PREDICTION_INPUTS_PER_DONOR,
    "scaler_cells_per_donor": SCALER_CELLS_PER_DONOR,
    "filter_counts": FILTER_COUNTS, "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS, "patience": EARLY_STOPPING_PATIENCE,
    "threshold": 0.5, "implementation_version": "pytorch_materialized_v1",
    "model_format": "complete_filter_parameters_v1",
}
expected_config = make_run_config(
    baseline_config["parameters"] | expected_parameters,
    [SPLITS_PATH, LABELS_PATH, MARKERS_PATH, *fcs_paths],
    PROJECT_ROOT / "notebooks/04c_cellcnn.ipynb", [4, 6, 8, 10],
)
if baseline_config != expected_config:
    raise ValueError("Baseline-Konfiguration, Referenzcode oder Eingangsdaten passen nicht.")

baseline_predictions = pd.read_csv(baseline_path, float_precision="round_trip")
baseline_filters_path = TABLES / "task4_cellcnn_filters_gated_alive_full.csv"
baseline_filters = pd.read_csv(baseline_filters_path, float_precision="round_trip")
baseline_selection = pd.read_csv(TABLES / "task4_cellcnn_selection_gated_alive_full.csv")
baseline_predictions = baseline_predictions.loc[baseline_predictions.split_id.isin(BONUS_SPLIT_IDS)].copy()
baseline_filters = baseline_filters.loc[baseline_filters.split_id.isin(BONUS_SPLIT_IDS)].copy()
validate_prediction_splits(baseline_predictions, donor_splits)
validate_parameter_table(baseline_filters, baseline_predictions, markers, ["split_id", "filter_id"],
    ["filter_weight", "filter_bias", "output_weight_0", "output_weight_1",
     "output_bias_0", "output_bias_1", "output_weight_contrast", "scaler_mean", "scaler_scale"])
assert set(baseline_predictions.split_id) == set(BONUS_SPLIT_IDS)
for split_id in BONUS_SPLIT_IDS:
    split = donor_splits.loc[donor_splits.split_id.eq(split_id)]
    assert len(split) == 20 and split.donor_id.nunique() == 20
    assert split.outer_partition.eq("test").sum() == 6
    assert split.outer_partition.eq("train").sum() == 14
    assert set(split.loc[split.outer_partition.eq("train"), "inner_fold"]) == {0, 1, 2}
    selection = baseline_selection.loc[baseline_selection.split_id.eq(split_id)]
    assert len(selection) == 9
    assert set(zip(selection.inner_fold, selection.filter_count)) == {(f, k) for f in range(3) for k in FILTER_COUNTS}
    best = selection.sort_values(
        ["validation_accuracy", "validation_roc_auc", "best_validation_loss", "filter_count", "inner_fold"],
        ascending=[False, False, True, True, True]).iloc[0]
    assert selection.selected.sum() == 1 and bool(best.selected)
    filters = baseline_filters.loc[baseline_filters.split_id.eq(split_id)]
    pred = baseline_predictions.loc[baseline_predictions.split_id.eq(split_id)]
    assert filters.inner_fold.eq(best.inner_fold).all() and pred.selected_inner_fold.eq(best.inner_fold).all()
    assert filters.filter_id.nunique() == best.filter_count and pred.filter_count.eq(best.filter_count).all()
    assert np.allclose(filters.output_weight_contrast, filters.output_weight_1 - filters.output_weight_0)
    assert filters.scaler_scale.gt(0).all()
    for frame in [filters, pred]:
        assert frame.gate.eq(GATE).all() and frame.run_mode.eq(RUN_MODE).all()
        assert frame.top_fraction.eq(TOP_FRACTION).all()
    assert pred.score.between(0, 1).all()

# Identische Originalereignisse je Spender, unabhängig von Modell und Split.
evaluation_indices = {
    donor: np.random.default_rng(EVALUATION_SEED + i).choice(
        len(data_by_donor[donor]), min(EVALUATION_CELLS_PER_DONOR, len(data_by_donor[donor])), replace=False)
    for i, donor in enumerate(sorted(data_by_donor))
}
evaluation_config = {
    "baseline_config": baseline_config,
    "baseline_predictions_sha256": file_digest(baseline_path),
    "baseline_filters_sha256": file_digest(baseline_filters_path),
    "split_ids": BONUS_SPLIT_IDS, "evaluation_seed": EVALUATION_SEED,
    "evaluation_cells_per_donor": EVALUATION_CELLS_PER_DONOR,
    "selection": "largest_positive_output_contrast", "baseline_membership": "half_inner_training_maximum",
    "event_indices_sha256": hashlib.sha256(b"".join(
        evaluation_indices[d].astype("<i8").tobytes() for d in sorted(evaluation_indices))).hexdigest(),
}
print("Baseline-Artefakte, Referenzcode und spenderweise Splits geprüft.")


Baseline-Artefakte, Referenzcode und spenderweise Splits geprüft.


## Modell und Trainingsverfahren

Die Trainingsfunktionen folgen 04c: spendergleiches Bag-Sampling, Trainings-Scaler, Adam, Lernrate, L2-Koeffizient, Top-1%-Pooling, maximal 100 Epochen und Early Stopping bleiben gleich. Pro Outer-Split werden drei innere Folds und die Filterzahlen 3, 4 und 5 verglichen. Die Auswahl erfolgt wie bisher nach Validierungsgenauigkeit, Validierungs-AUC, Verlust und festen Tie-Breakern. Das ausgewählte innere Modell wird direkt getestet; es findet kein Refit auf allen äußeren Trainingsspendern statt.

Die linearen Filter und der Output-Layer werden in derselben Reihenfolge wie in 04c initialisiert. Die anschließende Nullinitialisierung von `q` verbraucht keine Zufallszahlen. Die einzige zusätzliche Strafe ist `L2_COEFFICIENT * sum(q²)`; Biases bleiben wie bisher unregularisiert.

In [4]:
IMPLEMENTATION_VERSION = "quadratic_relu_top1_v1"


class QuadraticCellCNN(nn.Module):
    """Lineare plus diagonale quadratische Filterantwort mit ReLU und Top-1%."""
    def __init__(self, marker_count, filter_count, top_fraction=TOP_FRACTION):
        super().__init__()
        self.cell_filters = nn.Linear(marker_count, filter_count)
        self.output_layer = nn.Linear(filter_count, 2)
        self.quadratic_weights = nn.Parameter(torch.zeros(filter_count, marker_count))
        self.top_fraction = top_fraction

    def responses(self, values):
        linear = self.cell_filters(values)
        quadratic = nn.functional.linear(values.square(), self.quadratic_weights)
        return torch.relu(linear + quadratic)

    def forward(self, values):
        responses = self.responses(values)
        top_count = max(1, int(self.top_fraction * values.shape[1]))
        pooled = torch.topk(responses, k=top_count, dim=1).values.mean(dim=1)
        return self.output_layer(pooled)


In [5]:
# Zweck: Spenderbalancierte Multi-Cell-Inputs erzeugen und die CellCNN-Schichten definieren.
def materialize_multicell_inputs(
    donor_ids: list[str],
    scaled_data: dict[str, np.ndarray],
    cells_per_input: int,
    inputs_per_donor: int,
    seed: int,
) -> TensorDataset:
    """Feste zufällige Zellgruppen mit je einem Spenderlabel materialisieren.

    Jeder Spender erzeugt gleich viele Inputs. Ziehen mit Zurücklegen erhöht
    die Zahl der Trainingsbeispiele, ohne Spender als unabhängig zu vervielfachen.
    """
    donor_ids = sorted(donor_ids)
    input_count = len(donor_ids) * inputs_per_donor
    values = np.empty(
        (input_count, cells_per_input, len(markers)), dtype=np.float32
    )
    labels = np.empty(input_count, dtype=np.int64)
    input_index = 0
    for donor_id in donor_ids:
        donor_values = scaled_data[donor_id]
        for _ in range(inputs_per_donor):
            # Ein eigener deterministischer Teilseed macht jeden Input reproduzierbar.
            rng = np.random.default_rng(seed + input_index)
            cell_indices = rng.integers(
                0, len(donor_values), size=cells_per_input
            )
            values[input_index] = donor_values[cell_indices]
            labels[input_index] = label_by_donor[donor_id]
            input_index += 1
    return TensorDataset(
        torch.from_numpy(values).to(DEVICE),
        torch.from_numpy(labels).to(DEVICE),
    )


def fit_balanced_scaler(donor_ids: list[str], cells_per_donor: int, seed: int):
    """Scaler auf gleich vielen Zellen je Trainingsspender fitten."""
    rng = np.random.default_rng(seed)
    scaler_parts = []
    for donor_id in sorted(donor_ids):
        values = data_by_donor[donor_id]
        if len(values) < cells_per_donor:
            raise ValueError(f"Zu wenige Zellen für Skalierung: {donor_id}")
        indices = rng.choice(len(values), size=cells_per_donor, replace=False)
        scaler_parts.append(values[indices])
    scaler = StandardScaler().fit(np.concatenate(scaler_parts))
    return scaler


def scale_donors(donor_ids: list[str], scaler: StandardScaler):
    """Bereits gefittete Trainingsparameter unverändert auf Spender anwenden."""
    return {
        donor_id: scaler.transform(data_by_donor[donor_id]).astype(np.float32, copy=False)
        for donor_id in donor_ids
    }


In [6]:
# Zweck: Modellkandidaten mit L2-Regularisierung und validierungsbasiertem Stopp trainieren.
def regularized_loss(model: QuadraticCellCNN, logits: torch.Tensor, labels: torch.Tensor):
    """Kreuzentropie plus L2-Strafe der lernbaren Gewichtsmatrizen berechnen."""
    cross_entropy = nn.functional.cross_entropy(logits, labels)
    weight_penalty = (
        model.cell_filters.weight.square().sum()
        + model.quadratic_weights.square().sum()
        + model.output_layer.weight.square().sum()
    )
    return cross_entropy + L2_COEFFICIENT * weight_penalty


def evaluate_loader(model: QuadraticCellCNN, loader: DataLoader) -> float:
    """Mittleren regularisierten Verlust ohne Gradienten berechnen."""
    model.eval()
    weighted_loss_sum = 0.0
    example_count = 0
    with torch.no_grad():
        for values, labels in loader:
            logits = model(values.to(DEVICE))
            batch_loss = float(regularized_loss(model, logits, labels.to(DEVICE)))
            weighted_loss_sum += batch_loss * len(labels)
            example_count += len(labels)
    return weighted_loss_sum / example_count


def train_candidate(
    train_ids: list[str],
    validation_ids: list[str],
    filter_count: int,
    seed: int,
) -> tuple[QuadraticCellCNN, StandardScaler, dict[str, object]]:
    """Einen Kandidaten ausschließlich auf innerem Training und Validierung fitten.

    Zurückgegeben werden der beste Modellzustand, sein Trainings-Scaler und
    die Lernhistorie; äußere Testspender werden hier nie verwendet.
    """
    torch.manual_seed(seed)
    scaler = fit_balanced_scaler(train_ids, SCALER_CELLS_PER_DONOR, seed)
    scaled_data = scale_donors(train_ids + validation_ids, scaler)
    train_dataset = materialize_multicell_inputs(
        train_ids, scaled_data, TRAINING_CELLS_PER_INPUT,
        TRAINING_INPUTS_PER_DONOR, seed + 1_000,
    )
    validation_dataset = materialize_multicell_inputs(
        validation_ids, scaled_data, TRAINING_CELLS_PER_INPUT,
        TRAINING_INPUTS_PER_DONOR, seed + 2_000,
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, generator=generator,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    model = QuadraticCellCNN(len(markers), filter_count, TOP_FRACTION).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_state = copy.deepcopy(model.state_dict())
    best_validation_loss = np.inf
    epochs_without_improvement = 0
    history = []

    for epoch in range(MAX_EPOCHS):
        model.train()
        train_losses = []
        for values, labels in train_loader:
            optimizer.zero_grad()
            logits = model(values.to(DEVICE))
            loss = regularized_loss(model, logits, labels.to(DEVICE))
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach()))

        validation_loss = evaluate_loader(model, validation_loader)
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": float(np.mean(train_losses)),
                "validation_loss": validation_loss,
            }
        )
        # Nur echte Verbesserungen ersetzen den gesicherten besten Zustand.
        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                break

    model.load_state_dict(best_state)
    training_info = {
        "epochs_run": len(history),
        "best_validation_loss": best_validation_loss,
        "history": history,
    }
    return model, scaler, training_info

In [7]:
# Zweck: Kandidaten spenderweise bewerten, auswählen und ihre Filter für Aufgabe 5 sichern.
def predict_donor(
    model: QuadraticCellCNN,
    scaler: StandardScaler,
    donor_id: str,
    seed: int,
) -> float:
    """Mehrere zufällige Multi-Cell-Vorhersagen eines Spenders mitteln."""
    values = data_by_donor[donor_id]
    input_cell_count = min(PREDICTION_CELLS_PER_INPUT, len(values))
    rng = np.random.default_rng(seed)
    probabilities = []
    model.eval()
    with torch.no_grad():
        for _ in range(PREDICTION_INPUTS_PER_DONOR):
            indices = rng.choice(len(values), size=input_cell_count, replace=False)
            scaled = scaler.transform(values[indices]).astype(np.float32, copy=False)
            logits = model(torch.from_numpy(scaled).unsqueeze(0).to(DEVICE))
            probability = torch.softmax(logits, dim=1)[0, 1].item()
            probabilities.append(probability)
    return float(np.mean(probabilities))


def train_outer_split(split_id: int):
    """Alle inneren Kandidaten fitten und genau ein Modell extern testen.

    Die Funktion liefert getrennte Tabellen für Testvorhersagen, Auswahlprozess
    und interpretierbare Filterparameter des ausgewählten Netzes.
    """
    split = donor_splits.loc[donor_splits["split_id"] == split_id]
    split_seed = int(split["split_seed"].iloc[0])
    test_ids = split.loc[split["outer_partition"] == "test", "donor_id"].tolist()
    candidates = []
    selection_records = []

    for inner_fold in range(3):
        train_ids = split.loc[
            (split["outer_partition"] == "train")
            & (split["inner_fold"] != inner_fold), "donor_id"
        ].tolist()
        validation_ids = split.loc[
            (split["outer_partition"] == "train")
            & (split["inner_fold"] == inner_fold), "donor_id"
        ].tolist()
        assert not set(train_ids) & set(validation_ids)
        assert not set(train_ids + validation_ids) & set(test_ids)

        for filter_count in FILTER_COUNTS:
            candidate_seed = split_seed + 10_000 * inner_fold + 100 * filter_count
            model, scaler, training_info = train_candidate(
                train_ids, validation_ids, filter_count, candidate_seed
            )
            validation_scores = np.array(
                [
                    predict_donor(model, scaler, donor_id, candidate_seed + 50_000 + index)
                    for index, donor_id in enumerate(sorted(validation_ids))
                ]
            )
            validation_true = np.array(
                [label_by_donor[x] for x in sorted(validation_ids)]
            )
            validation_accuracy = float(
                ((validation_scores >= 0.5).astype(int) == validation_true).mean()
            )
            validation_auc = float(roc_auc_score(validation_true, validation_scores))
            record = {
                "split_id": split_id,
                "gate": GATE,
                "run_mode": RUN_MODE,
                "device": DEVICE.type,
                "implementation_version": IMPLEMENTATION_VERSION,
                "inner_fold": inner_fold,
                "filter_count": filter_count,
                "candidate_seed": candidate_seed,
                "validation_accuracy": validation_accuracy,
                "validation_roc_auc": validation_auc,
                "best_validation_loss": training_info["best_validation_loss"],
                "epochs_run": training_info["epochs_run"],
            }
            selection_records.append(record)
            candidates.append({"model": model, "scaler": scaler, **record})
            print(f"  Fold {inner_fold}, Filter {filter_count}: {training_info['epochs_run']} Epochen", flush=True)

    # Sortierschlüssel kodiert die vorab festgelegte Auswahl samt Tie-Breakern.
    candidates.sort(
        key=lambda item: (
            -item["validation_accuracy"],
            -item["validation_roc_auc"],
            item["best_validation_loss"],
            item["filter_count"],
            item["inner_fold"],
        )
    )
    selected = candidates[0]
    for record in selection_records:
        record["selected"] = (
            record["inner_fold"] == selected["inner_fold"]
            and record["filter_count"] == selected["filter_count"]
        )

    prediction_records = []
    for index, donor_id in enumerate(sorted(test_ids)):
        score = predict_donor(
            selected["model"], selected["scaler"], donor_id,
            split_seed + 900_000 + index,
        )
        prediction_records.append(
            {
                "method": "cellcnn_quadratic",
                "gate": GATE,
                "run_mode": RUN_MODE,
                "device": DEVICE.type,
                "implementation_version": IMPLEMENTATION_VERSION,
                "split_id": split_id,
                "split_seed": split_seed,
                "donor_id": donor_id,
                "y_true": label_by_donor[donor_id],
                "score": score,
                "decision_threshold": 0.5,
                "y_pred": int(score >= 0.5),
                "filter_count": selected["filter_count"],
                "selected_inner_fold": selected["inner_fold"],
                "training_cells_per_input": TRAINING_CELLS_PER_INPUT,
                "training_inputs_per_donor": TRAINING_INPUTS_PER_DONOR,
                "prediction_cells_per_input": min(
                    PREDICTION_CELLS_PER_INPUT, len(data_by_donor[donor_id])
                ),
                "prediction_inputs_per_donor": PREDICTION_INPUTS_PER_DONOR,
                "pooling": "top_fraction_mean_relu_responses",
                "top_fraction": TOP_FRACTION,
            }
        )

    return pd.DataFrame(prediction_records), pd.DataFrame(selection_records), selected


## Kleiner Smoke-Test vor dem Full-Lauf

Zuerst werden Formel, Initialisierung und Gradienten synthetisch geprüft. Danach durchläuft ein kleiner echter Outer-Split alle neun Kandidaten mit zwei Epochen. Dessen Ergebnisse werden weder gespeichert noch mit Full-Ergebnissen vermischt. Sämtliche Full-Einstellungen werden danach wiederhergestellt.

In [8]:
def smoke_test():
    rng = np.random.default_rng(601)
    values = torch.from_numpy(rng.normal(size=(2, 512, len(markers))).astype(np.float32)).to(DEVICE)
    for filter_count in FILTER_COUNTS:
        torch.manual_seed(601)
        linear_filters = nn.Linear(len(markers), filter_count).to(DEVICE)
        linear_output = nn.Linear(filter_count, 2).to(DEVICE)
        expected_rng = torch.rand(4)
        torch.manual_seed(601)
        model = QuadraticCellCNN(len(markers), filter_count).to(DEVICE)
        assert torch.equal(expected_rng, torch.rand(4))
        assert torch.count_nonzero(model.quadratic_weights) == 0
        for name in ["weight", "bias"]:
            assert torch.equal(getattr(linear_filters, name), getattr(model.cell_filters, name))
            assert torch.equal(getattr(linear_output, name), getattr(model.output_layer, name))
        # Sortierreferenz prüft Baseline-Gleichheit, Abrunden und mindestens eine Zelle.
        for count in [512, 257, 51]:
            subset = values[:, :count]
            responses = torch.relu(linear_filters(subset))
            top_count = max(1, int(TOP_FRACTION * count))
            expected = linear_output(responses.sort(dim=1, descending=True).values[:, :top_count].mean(1))
            torch.testing.assert_close(model(subset), expected)
        assert sum(p.numel() for p in model.parameters()) == (
            sum(p.numel() for p in linear_filters.parameters())
            + sum(p.numel() for p in linear_output.parameters()) + filter_count * len(markers))

    labels = torch.tensor([0, 1], device=DEVICE)
    logits = model(values)
    loss = regularized_loss(model, logits, labels)
    expected_penalty = (model.cell_filters.weight.square().sum()
                        + model.quadratic_weights.square().sum()
                        + model.output_layer.weight.square().sum())
    torch.testing.assert_close(loss, nn.functional.cross_entropy(logits, labels) + L2_COEFFICIENT * expected_penalty)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
    assert torch.count_nonzero(model.quadratic_weights.grad) > 0
    optimizer.step()
    assert torch.count_nonzero(model.quadratic_weights) > 0
    # Auch die nichtlineare Antwort nach dem Update gegen die explizite Formel prüfen.
    explicit = torch.relu((values[:, :, None, :] * model.cell_filters.weight
                          + values[:, :, None, :].square() * model.quadratic_weights).sum(-1)
                         + model.cell_filters.bias)
    torch.testing.assert_close(model.responses(values), explicit, atol=1e-5, rtol=1e-5)
    torch.testing.assert_close(model(values), model(values.flip(1)))
    fixed_logits = torch.zeros_like(logits)
    loss_before = regularized_loss(model, fixed_logits, labels)
    old_q = model.quadratic_weights.detach().clone()
    with torch.no_grad():
        model.quadratic_weights.add_(1)
    penalty_change = L2_COEFFICIENT * (model.quadratic_weights.square() - old_q.square()).sum()
    torch.testing.assert_close(regularized_loss(model, fixed_logits, labels) - loss_before, penalty_change)
    print("Baseline-Gleichheit bei q=0, Zufallsfolge, Top-1%, quadratische Formel, L2 und Lernen geprüft.")

    # Ein kompletter kleiner Outer-Split mit echter Auswahl-/Vorhersagelogik; keine Exporte.
    small = {"TRAINING_CELLS_PER_INPUT": 256, "TRAINING_INPUTS_PER_DONOR": 4,
             "PREDICTION_CELLS_PER_INPUT": 1000, "PREDICTION_INPUTS_PER_DONOR": 2,
             "SCALER_CELLS_PER_DONOR": 1000, "BATCH_SIZE": 16, "MAX_EPOCHS": 2}
    original = {key: globals()[key] for key in small}
    try:
        globals().update(small)
        predictions, selection, chosen = train_outer_split(BONUS_SPLIT_IDS[0])
        validate_prediction_splits(predictions, donor_splits)
        assert len(selection) == 9 and selection.selected.sum() == 1
        assert predictions.score.between(0, 1).all()
        print("Isolierter Smoke-Outer-Split bestanden; kein Full-Ergebnis:",
              roc_auc_score(predictions.y_true, predictions.score))
    finally:
        globals().update(original)

smoke_test()


Baseline-Gleichheit bei q=0, Zufallsfolge, Top-1%, quadratische Formel, L2 und Lernen geprüft.


  Fold 0, Filter 3: 2 Epochen


  Fold 0, Filter 4: 2 Epochen


  Fold 0, Filter 5: 2 Epochen


  Fold 1, Filter 3: 2 Epochen


...

  Fold 1, Filter 5: 2 Epochen


  Fold 2, Filter 3: 2 Epochen


  Fold 2, Filter 4: 2 Epochen


  Fold 2, Filter 5: 2 Epochen


Isolierter Smoke-Outer-Split bestanden; kein Full-Ergebnis: 0.75


In [9]:
def strongest_positive_filter(model):
    contrasts = (model.output_layer.weight[1] - model.output_layer.weight[0]).detach().cpu().numpy()
    positive = np.flatnonzero(contrasts > 0)
    return int(positive[np.argmax(contrasts[positive])]) if len(positive) else None


def summarize_split(predictions, frequencies, split_id):
    pred = predictions.loc[predictions.split_id.eq(split_id)]
    freq = frequencies.loc[frequencies.split_id.eq(split_id)]
    valid = len(freq) == len(pred) and freq.frequency.notna().all()
    return {
        "split_id": split_id,
        "network_auc": roc_auc_score(pred.y_true, pred.score),
        "frequency_auc": roc_auc_score(freq.y_true, freq.frequency) if valid else np.nan,
        "frequency_effect": (freq.loc[freq.y_true.eq(1), "frequency"].mean()
                             - freq.loc[freq.y_true.eq(0), "frequency"].mean()) if valid else np.nan,
        "phenotype_status": "berechnet" if valid else "kein positiver Output-Kontrast",
    }


## Full-Ausführung und Wiederaufnahme

Die zehn festgelegten Splits werden nacheinander gerechnet. `TASK6_SPLIT_LIMIT` kann einen technischen Teillauf begrenzen; die Standardauswertung verwendet alle zehn Splits. `TASK6_RUN_TRAINING=0` verlangt vorhandene passende Checkpoints. Der kleine Smoke-Test läuft weiterhin vorab.

Nach jedem vollständigen Split werden Modell einschließlich `q`, Scaler, Kandidatenauswahl und Donorvorhersagen unter einem eigenen Dateinamen gespeichert. Referenzcode, Eingabedaten und Hyperparameter werden vor Wiederverwendung geprüft.

Für beide Methoden wird der Filter mit größtem positiven Output-Kontrast ausgewählt. Seine Phänotyppopulation erfüllt die Halbmaximum-Regel: Antwort größer als die Hälfte des Maximums aus den tatsächlichen inneren Trainingsspendern dieses Modells. Beide Methoden evaluieren dieselben bis zu 20.000 Originalzellen je Testspender. Ohne positiven Filter bleiben die Phänotypmetriken fehlend; eine leere Population hat Häufigkeit null. Baseline-Ergebnisse werden aus Task-4-Artefakten rekonstruiert, ohne bestehende Bonus-Exporte zu benötigen.

In [10]:
TRAINING_CODE_CELLS = [1, 2, 5, 6, 7, 8]
# Standard: zehn Splits. TASK6_SPLIT_LIMIT=1 erlaubt zunächst nur den ersten Lauf.
split_limit = int(os.environ.get("TASK6_SPLIT_LIMIT", "0"))
active_split_ids = BONUS_SPLIT_IDS[:split_limit] if split_limit > 0 else BONUS_SPLIT_IDS
run_training = os.environ.get("TASK6_RUN_TRAINING", "1") == "1"
frequency_rows, check_rows = [], []
for split_id in active_split_ids:
    split = donor_splits.loc[donor_splits.split_id.eq(split_id)]
    filters = baseline_filters.loc[baseline_filters.split_id.eq(split_id)]
    model = SavedCellCNN(filters, markers).to(DEVICE)
    filter_id = strongest_positive_filter(model)
    train_ids = sorted(split.loc[split.outer_partition.eq("train") &
                                 split.inner_fold.ne(int(filters.inner_fold.iloc[0])), "donor_id"])
    test_ids = sorted(split.loc[split.outer_partition.eq("test"), "donor_id"])
    assert len(train_ids) in (9, 10) and not set(train_ids) & set(test_ids)
    maximum = 0.0
    with torch.no_grad():
        if filter_id is not None:
            for donor in train_ids:
                values = data_by_donor[donor]
                for start in range(0, len(values), 50000):
                    scaled = model.scaler.transform(values[start:start + 50000])
                    response = torch.relu(model.cell_filters(torch.from_numpy(scaled).to(DEVICE)))
                    maximum = max(maximum, float(response[:, filter_id].max()))
        for i, donor in enumerate(test_ids):
            # Originale Prediction-Bags aus 04c rekonstruieren, keine neuen Vorhersageparameter.
            values = data_by_donor[donor]
            rng = np.random.default_rng(int(split.split_seed.iloc[0]) + 900000 + i)
            probabilities = []
            for _ in range(PREDICTION_INPUTS_PER_DONOR):
                indices = rng.choice(len(values), min(PREDICTION_CELLS_PER_INPUT, len(values)), replace=False)
                bag = torch.from_numpy(model.scaler.transform(values[indices])).to(DEVICE)
                logits, _ = model(bag)
                probabilities.append(float(torch.softmax(logits, dim=0)[1]))
            expected = baseline_predictions.loc[baseline_predictions.split_id.eq(split_id) & baseline_predictions.donor_id.eq(donor)].iloc[0]
            error = abs(float(np.mean(probabilities)) - expected.score)
            assert error <= 1e-6, (split_id, donor, error)
            check_rows.append({"split_id": split_id, "donor_id": donor, "probability_error": error})
            indices = evaluation_indices[donor]
            frequency = np.nan
            if filter_id is not None:
                scaled = torch.from_numpy(model.scaler.transform(values[indices])).to(DEVICE)
                response = torch.relu(model.cell_filters(scaled))[:, filter_id]
                frequency = float((response > maximum / 2).float().mean()) if maximum > 0 else 0.0
            frequency_rows.append({"split_id": split_id, "donor_id": donor, "y_true": label_by_donor[donor],
                "filter_id": filter_id, "frequency": frequency, "n_cells": len(indices),
                "response_threshold": maximum / 2 if filter_id is not None else np.nan,
                "training_donors": ";".join(train_ids)})
    print(f"Baseline-Split {split_id}: geladen; Vorhersagen rekonstruiert.")

baseline_frequencies = pd.DataFrame(frequency_rows)
baseline_metrics = pd.DataFrame([summarize_split(baseline_predictions, baseline_frequencies, s) for s in active_split_ids])
pd.DataFrame(check_rows).to_csv(TABLES / "task6_quadratic_baseline_prediction_checks.csv", index=False)
baseline_frequencies.to_csv(TABLES / "task6_quadratic_baseline_frequencies.csv", index=False)
baseline_metrics.to_csv(TABLES / "task6_quadratic_baseline_metrics.csv", index=False)

run_config = make_run_config(
    expected_parameters | {
        "method": "cellcnn_quadratic", "implementation_version": IMPLEMENTATION_VERSION,
        "model_format": "state_dict_and_scaler_v1", "top_fraction": TOP_FRACTION,
        "pooling": "top_fraction_mean_relu_responses", "l2_scope": "linear_quadratic_output_weights",
        "quadratic_initialization": "zeros", "device": str(DEVICE),
        "numpy": version("numpy"), "torch": version("torch"),
        "scikit_learn": version("scikit-learn"), "flowkit": version("flowkit"),
    }, [SPLITS_PATH, LABELS_PATH, MARKERS_PATH, *fcs_paths],
    PROJECT_ROOT / "notebooks/06d_cellcnn_quadratic.ipynb", TRAINING_CODE_CELLS,
)
frequency_rows, prediction_parts, selection_parts, timing_rows = [], [], [], []
for split_id in active_split_ids:
    checkpoint_path = TABLES / f"task6_quadratic_split_{split_id}.pt"
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        if checkpoint["config"] != run_config or checkpoint["split_id"] != split_id:
            raise ValueError(f"Unpassender Bonus-Checkpoint: {checkpoint_path.name}; nicht überschrieben.")
        print(f"Quadratischer Split {split_id}: passenden Checkpoint geladen.", flush=True)
    else:
        if not run_training:
            raise FileNotFoundError(f"Fehlender Bonus-Checkpoint: {checkpoint_path.name}")
        started = time.monotonic()
        predictions, selection, selected = train_outer_split(split_id)
        checkpoint = {
            "config": run_config, "split_id": split_id,
            "state_dict": {name: value.detach().cpu() for name, value in selected["model"].state_dict().items()},
            "scaler_mean": selected["scaler"].mean_.tolist(), "scaler_scale": selected["scaler"].scale_.tolist(),
            "filter_count": int(selected["filter_count"]), "inner_fold": int(selected["inner_fold"]),
            "predictions": predictions.to_dict("records"), "selection": selection.to_dict("records"),
            "training_seconds": time.monotonic() - started,
        }
        temporary = checkpoint_path.with_suffix(".tmp")
        torch.save(checkpoint, temporary)
        temporary.replace(checkpoint_path)
        del selected
        print(f"Quadratischer Split {split_id}: {checkpoint['training_seconds'] / 60:.1f} Minuten; gespeichert.", flush=True)

    predictions = pd.DataFrame(checkpoint["predictions"])
    selection = pd.DataFrame(checkpoint["selection"])
    validate_prediction_splits(predictions, donor_splits)
    assert set(predictions.split_id) == {split_id} and predictions.score.between(0, 1).all()
    assert len(selection) == 9 and selection.selected.sum() == 1
    best = selection.sort_values(
        ["validation_accuracy", "validation_roc_auc", "best_validation_loss", "filter_count", "inner_fold"],
        ascending=[False, False, True, True, True]).iloc[0]
    assert bool(best.selected) and best.inner_fold == checkpoint["inner_fold"] and best.filter_count == checkpoint["filter_count"]
    scaler = restore_scaler(pd.DataFrame({"scaler_mean": checkpoint["scaler_mean"], "scaler_scale": checkpoint["scaler_scale"]}))
    model = QuadraticCellCNN(len(markers), checkpoint["filter_count"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    assert all(torch.isfinite(p).all() for p in model.parameters())
    filter_id = strongest_positive_filter(model)
    split = donor_splits.loc[donor_splits.split_id.eq(split_id)]
    test_ids = sorted(split.loc[split.outer_partition.eq("test"), "donor_id"])
    train_ids = sorted(split.loc[split.outer_partition.eq("train") &
                                 split.inner_fold.ne(checkpoint["inner_fold"]), "donor_id"])
    assert len(train_ids) in (9, 10) and not set(train_ids) & set(test_ids)
    maximum = 0.0
    with torch.no_grad():
        if filter_id is not None:
            for donor in train_ids:
                values = data_by_donor[donor]
                for start in range(0, len(values), 50000):
                    scaled = torch.from_numpy(scaler.transform(values[start:start + 50000])).to(DEVICE)
                    maximum = max(maximum, float(model.responses(scaled)[:, filter_id].max()))
        for i, donor in enumerate(test_ids):
            score = predict_donor(model, scaler, donor, int(split.split_seed.iloc[0]) + 900000 + i)
            expected_score = predictions.loc[predictions.donor_id.eq(donor), "score"].item()
            assert abs(score - expected_score) <= 1e-6
            indices = evaluation_indices[donor]
            frequency = np.nan
            if filter_id is not None:
                values = torch.from_numpy(scaler.transform(data_by_donor[donor][indices])).to(DEVICE)
                frequency = float((model.responses(values)[:, filter_id] > maximum / 2).float().mean()) if maximum > 0 else 0.0
            frequency_rows.append({"split_id": split_id, "donor_id": donor, "y_true": label_by_donor[donor],
                                   "filter_id": filter_id, "frequency": frequency, "n_cells": len(indices),
                                   "response_threshold": maximum / 2 if filter_id is not None else np.nan,
                                   "training_donors": ";".join(train_ids)})
    prediction_parts.append(predictions)
    selection_parts.append(selection)
    timing_rows.append({"split_id": split_id, "training_seconds": checkpoint["training_seconds"]})
    modified_predictions = pd.concat(prediction_parts, ignore_index=True)
    modified_frequencies = pd.DataFrame(frequency_rows)
    modified_predictions.to_csv(TABLES / "task6_quadratic_predictions.csv", index=False)
    modified_frequencies.to_csv(TABLES / "task6_quadratic_frequencies.csv", index=False)
    pd.concat(selection_parts, ignore_index=True).to_csv(TABLES / "task6_quadratic_selection.csv", index=False)
    pd.DataFrame(timing_rows).to_csv(TABLES / "task6_quadratic_timing.csv", index=False)

modified_metrics = pd.DataFrame([summarize_split(modified_predictions, modified_frequencies, s) for s in active_split_ids])
modified_metrics.to_csv(TABLES / "task6_quadratic_metrics.csv", index=False)


Baseline-Split 0: geladen; Vorhersagen rekonstruiert.


Baseline-Split 1: geladen; Vorhersagen rekonstruiert.


Baseline-Split 2: geladen; Vorhersagen rekonstruiert.
Baseline-Split 3: geladen; Vorhersagen rekonstruiert.


Baseline-Split 4: geladen; Vorhersagen rekonstruiert.
Baseline-Split 5: geladen; Vorhersagen rekonstruiert.


Baseline-Split 6: geladen; Vorhersagen rekonstruiert.
Baseline-Split 7: geladen; Vorhersagen rekonstruiert.


...

Quadratischer Split 5: passenden Checkpoint geladen.


Quadratischer Split 6: passenden Checkpoint geladen.


Quadratischer Split 7: passenden Checkpoint geladen.


Quadratischer Split 8: passenden Checkpoint geladen.


Quadratischer Split 9: passenden Checkpoint geladen.


## Gepaarter Vergleich auf denselben zehn Splits

Primäre Metrik ist die Network-ROC-AUC pro Outer-Split. Die Tabellen zeigen beide AUCs und ihre gepaarte Differenz sowie Mittelwert und Median. Die Phänotyphäufigkeiten ergänzen die Klassifikation, ersetzen aber keine Zelltyp-Ground-Truth. Fehlende Phänotypmetriken und ihre Anzahl werden ausgewiesen.

Ein Testsplit umfasst zwei positive und vier negative Spender; ohne Rangbindungen ändert sich seine AUC daher in Schritten von 0,125. Spender wiederholen sich zwischen Splits und sind keine neuen unabhängigen Beobachtungen. Die älteren Drei-Split-Ergebnisse werden nicht mit diesem Zehn-Split-Mittel verglichen.

In [11]:
comparison = baseline_metrics.merge(modified_metrics, on="split_id", suffixes=("_baseline", "_modified"), validate="one_to_one")
assert set(comparison.split_id) == set(active_split_ids)
comparison["network_auc_delta"] = comparison.network_auc_modified - comparison.network_auc_baseline
# Der Vergleich enthält exakt dieselben Spender und Zellzahlen je Split.
keys = ["split_id", "donor_id", "y_true", "n_cells"]
paired = baseline_frequencies.merge(modified_frequencies, on=keys, validate="one_to_one")
assert len(paired) == len(modified_frequencies) == 6 * len(active_split_ids)
display(comparison[["split_id", "network_auc_baseline", "network_auc_modified", "network_auc_delta"]].rename(columns={
    "network_auc_baseline": "Baseline-AUC", "network_auc_modified": "Quadratische AUC", "network_auc_delta": "AUC-Differenz"}))
summary = pd.DataFrame({
    "Metrik": ["Mittlere Network-ROC-AUC", "Mediane Network-ROC-AUC", "Mittlere Häufigkeits-ROC-AUC",
               "Mittlere Häufigkeitsdifferenz CMV+ minus CMV−"],
    "Baseline": [comparison.network_auc_baseline.mean(), comparison.network_auc_baseline.median(),
                 comparison.frequency_auc_baseline.mean(), comparison.frequency_effect_baseline.mean()],
    "Quadratisches CellCNN": [comparison.network_auc_modified.mean(), comparison.network_auc_modified.median(),
                 comparison.frequency_auc_modified.mean(), comparison.frequency_effect_modified.mean()],
})
display(summary)
print(f"Mittlere AUC-Differenz: {comparison.network_auc_delta.mean():.4f}; Median: {comparison.network_auc_delta.median():.4f}")
print(f"Ausgewertete Full-Splits: {active_split_ids}; vorgesehen: {BONUS_SPLIT_IDS}")
print("Bestimmbare Phänotypmetriken:",
      {name: int(comparison[f"frequency_auc_{name}"].notna().sum()) for name in ["baseline", "modified"]})
display(comparison[["split_id", "frequency_auc_baseline", "frequency_auc_modified",
                    "frequency_effect_baseline", "frequency_effect_modified", "phenotype_status_baseline", "phenotype_status_modified"]])
comparison.to_csv(TABLES / "task6_quadratic_paired_comparison.csv", index=False)
summary.to_csv(TABLES / "task6_quadratic_comparison_summary.csv", index=False)


,split_id,Baseline-AUC,Quadratische AUC,AUC-Differenz
0,0,1.000,1.000,0.000
1,1,1.000,1.000,0.000
2,2,1.000,1.000,0.000
3,3,1.000,1.000,0.000
4,4,0.875,1.000,0.125
5,5,1.000,1.000,0.000
6,6,0.625,0.875,0.250
7,7,0.750,0.625,-0.125
8,8,0.500,0.875,0.375
9,9,1.000,1.000,0.000


,Metrik,Baseline,Quadratisches CellCNN
0,Mittlere Network-ROC-AUC,0.875000,0.937500
1,Mediane Network-ROC-AUC,1.000000,1.000000
2,Mittlere Häufigkeits-ROC-AUC,0.906250,0.837500
3,Mittlere Häufigkeitsdifferenz CMV+ minus CMV−,0.010735,0.005537


Mittlere AUC-Differenz: 0.0625; Median: 0.0000
Ausgewertete Full-Splits: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]; vorgesehen: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Bestimmbare Phänotypmetriken: {'baseline': 10, 'modified': 10}


,split_id,frequency_auc_baseline,frequency_auc_modified,frequency_effect_baseline,frequency_effect_modified,phenotype_status_baseline,phenotype_status_modified
0,0,1.0000,1.0000,0.007350,0.007587,berechnet,berechnet
1,1,1.0000,1.0000,0.033825,0.007237,berechnet,berechnet
2,2,1.0000,0.6875,0.026437,0.004675,berechnet,berechnet
3,3,1.0000,1.0000,0.009650,0.012687,berechnet,berechnet
4,4,0.8750,0.8750,0.001888,0.000800,berechnet,berechnet
5,5,1.0000,1.0000,0.012638,0.012937,berechnet,berechnet
6,6,0.5000,0.5000,0.003000,0.003912,berechnet,berechnet
7,7,0.7500,0.6250,0.000462,0.000012,berechnet,berechnet
8,8,0.9375,0.7500,0.008875,0.004488,berechnet,berechnet
9,9,1.0000,0.9375,0.003225,0.001037,berechnet,berechnet


Dieser Vergleich prüft eine quadratische Erweiterung der Filterantwort bei unverändertem Top-1%-Pooling. Zusätzliche Ausdrucksmöglichkeiten können helfen oder überanpassen. Eine enthaltene lineare Baseline garantiert keine bessere Testleistung.

Die Hypothese entstand nach Sichtung früherer Ergebnisse mit denselben 20 Spendern. Auch die größere, vor diesem Lauf festgelegte Splitmenge macht den Versuch deshalb nicht zu einer unabhängigen Bestätigung. Alle zehn Splits werden berichtet; es werden keine günstigeren Splits oder zusätzlichen Varianten nachträglich ausgewählt.